# WDL — Module 2 · Session 7 — Stations, grids and satellites
### Reading notebook

| | |
|---|---|
| **Course** | Hydroinformatics in a Changing Climate: Water Data Literacy for All |
| **Department** | Civil Engineering · CHRIST (Deemed to be University) |
| **Instructor** | Dr Arpan Pradhan |
| **Module** | Module 2 — Finding and Trusting Water Data |
| **Session** | Session 7 of 30 |
| **CO** | CO1, CO2 |
| **Topics** | Point gauges vs gridded rainfall vs satellite estimates; spatial resolution; strengths and limits of each |
| **Water hook** | *Same storm, three different numbers* |

---
> **How to use this notebook.** No coding needed. Read each explanation, then press ▶ on the box below it
> (or click the box and press `Shift + Enter`). Some boxes have sliders, number boxes or dropdowns: change them, then press ▶ again.
> Go to **File → Save a copy in Drive** first, so the notebook is yours.

## Why this session matters
The same storm can be reported as 80 mm by a gauge, 55 mm by a grid cell and 65 mm by a satellite, and none of them is
"wrong". They measure different things. This session shows why, so you can choose and explain your source.

In [ ]:
#@title ▶ Step 0 · Get ready (press ▶ once before anything else)
import os, json, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
_orig_bar = plt.Axes.bar
def _bar_int_years(self, x, *a, **k):        # whole-number year ticks on bar charts of years
    out = _orig_bar(self, x, *a, **k)
    try:
        xs = np.asarray(x)
        if xs.dtype.kind in "iu" and xs.min() > 1800: self.xaxis.set_major_locator(MaxNLocator(integer=True))
    except Exception: pass
    return out
plt.Axes.bar = _bar_int_years
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110, "font.size": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, GREY, TEAL, GOLD = "#2C6E9B", "#DD8452", "#C44E52", "#8C8C8C", "#1F7A8C", "#C9962E"
AUTO = os.environ.get("WDL_SELFTEST") == "1"

def ok(msg="Correct!"):  display(Markdown(f"✅ **{msg}**"))
def no(msg="Not quite. Check your working and try again."): display(Markdown(f"❌ {msg}"))
def _num(x):
    try: return float(str(x).replace(",", "").replace("%", "").strip())
    except (TypeError, ValueError): return None
def close(a, b, tol=0.02):
    return a is not None and abs(a - b) <= max(abs(b) * tol, 0.01)
def check_num(ans, exp, tol=0.02, fmt="{:,.1f}", hint=""):
    if AUTO: ans = exp
    v = _num(ans)
    if v is None: no("Type your answer in the box first (numbers only)."); return False
    if close(v, float(exp), tol): ok("Correct: " + fmt.format(exp)); return True
    no("Not quite. " + hint); return False
def check_choice(ans, exp, explain="", hint=""):
    if AUTO: ans = exp
    if ans in ("", "choose"): no("Choose an answer from the dropdown first."); return False
    if ans == exp: ok(("Correct. " + explain).strip()); return True
    no(("Not quite. " + hint).strip()); return False

def make_sample_district():
    """Illustrative, synthetic daily rainfall for a South Indian district, 1981-2024."""
    rng = np.random.default_rng(1981)
    days = pd.date_range("1981-01-01", "2024-12-31", freq="D")
    p_wet = np.array([0.03, 0.03, 0.06, 0.15, 0.30, 0.35, 0.42, 0.42, 0.45, 0.45, 0.28, 0.08])
    mu = np.array([4, 4, 7, 11, 13, 8, 8, 9, 13, 14, 12, 6], float)
    years = np.arange(1981, 2025)
    yf = dict(zip(years, np.exp(rng.normal(0, 0.17, len(years)))))
    m = days.month.values - 1
    t = days.year.values - 1981
    yfac = np.array([yf[y] for y in days.year.values])
    p = np.clip(p_wet[m] * (1 - 0.0025 * t) * yfac ** 0.5, 0, 0.95)
    inten = mu[m] * (1 + 0.005 * t) * yfac ** 0.5
    wet = rng.random(len(days)) < p
    amt = rng.gamma(0.7, inten / 0.7)
    rain = np.round(np.where(wet, amt, 0.0), 1)
    for d, v in [("1991-11-14", 142.0), ("2005-10-24", 186.2), ("2017-08-14", 151.4), ("2022-09-05", 168.9)]:
        rain[days.get_loc(pd.Timestamp(d))] = v
    return pd.Series(rain, index=days, name="rain_mm")

def make_messy_record(sample):
    """Three years of the sample record with typical gauge-record problems added."""
    df = sample["2018":"2020"].rename("rain_mm").reset_index()
    df.columns = ["date", "rain_mm"]
    flag_dates = ["2018-02-03", "2018-05-19", "2018-08-07", "2018-11-25", "2019-01-14",
                  "2019-04-02", "2019-09-18", "2019-12-30", "2020-06-06", "2020-10-11"]
    blank_dates = ["2018-07-21", "2019-03-09", "2020-02-17", "2020-08-28"]
    df.loc[df.date.isin(pd.to_datetime(flag_dates)), "rain_mm"] = -999.0
    df.loc[df.date.isin(pd.to_datetime(blank_dates)), "rain_mm"] = np.nan
    df.loc[df.date == "2018-09-12", "rain_mm"] = 999.9      # typing error
    df.loc[df.date == "2020-04-23", "rain_mm"] = -3.0       # impossible negative
    df = df[~df.date.between("2019-07-01", "2019-07-12")]  # 12 days never recorded
    dup = df[df.date.isin(pd.to_datetime(["2018-06-15", "2019-10-03", "2020-11-20"]))]
    df = pd.concat([df, dup]).sort_values("date", kind="stable").reset_index(drop=True)
    df["date"] = df.date.dt.strftime("%Y-%m-%d")
    return df

SAMPLE_URL = "https://raw.githubusercontent.com/PradhanArpan/waterliteracy/main/data/sample_district_daily_1981_2024.csv"
try:
    sample = pd.read_csv(SAMPLE_URL, parse_dates=["date"], index_col="date")["rain_mm"]
except Exception:
    sample = make_sample_district()   # works offline too: same numbers

def find_place(name):
    """Place name -> (lat, lon, label) using Open-Meteo geocoding. Prefers places in India."""
    parts = [p.strip() for p in str(name).split(",") if p.strip()]
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": parts[0], "count": 10, "language": "en", "format": "json"}, timeout=20)
    res = r.json().get("results", []) or []
    res = [x for x in res if x.get("country_code") == "IN"] or res
    if len(parts) > 1:
        res = [x for x in res if parts[1].lower() in str(x.get("admin1", "")).lower()] or res
    if not res: raise ValueError(f"No place found for '{name}'")
    x = res[0]
    return x["latitude"], x["longitude"], f'{x["name"]}, {x.get("admin1", "")}'.strip(", ")

def fetch_power(lat, lon, start=1981, end=2024):
    """Daily rainfall (mm/day) from NASA POWER (PRECTOTCORR) for one point."""
    r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", timeout=90, params={
        "parameters": "PRECTOTCORR", "community": "AG", "latitude": round(lat, 4), "longitude": round(lon, 4),
        "start": f"{start}0101", "end": f"{end}1231", "format": "JSON"})
    r.raise_for_status()
    d = r.json()["properties"]["parameter"]["PRECTOTCORR"]
    s = pd.Series(d, dtype=float)
    s.index = pd.to_datetime(s.index, format="%Y%m%d")
    s[s < -98] = np.nan
    return s.round(1).rename("rain_mm")

def fetch_era5(lat, lon, start="2021-01-01", end="2021-12-31"):
    """Daily rainfall (mm/day) from the ERA5 reanalysis via Open-Meteo."""
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params={
        "latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
        "daily": "precipitation_sum", "timezone": "Asia/Kolkata"})
    r.raise_for_status()
    d = r.json()["daily"]
    return pd.Series(d["precipitation_sum"], index=pd.to_datetime(d["time"]), dtype=float).rename("rain_mm")

def load_rain(source, place="", lat=0.0, lon=0.0):
    """Returns (daily rainfall Series, label, lat, lon). Falls back to the sample district."""
    if source.startswith("Sample") or AUTO:
        return sample, "Sample district (synthetic)", None, None
    try:
        if not (lat and lon):
            lat, lon, place = find_place(place)
        cache = f"rain_{lat:.3f}_{lon:.3f}.csv"
        if os.path.exists(cache):
            s = pd.read_csv(cache, parse_dates=["date"], index_col="date")["rain_mm"]
        else:
            s = fetch_power(lat, lon)
            s.rename_axis("date").to_csv(cache)
        return s, f"{place} ({lat:.2f}°N, {lon:.2f}°E) · NASA POWER", lat, lon
    except Exception as e:
        display(Markdown(f"⚠️ Could not download data for **{place or 'your place'}** ({type(e).__name__}). "
                         "Using the sample district instead. Check the place name or type latitude and longitude."))
        return sample, "Sample district (synthetic)", None, None

def annual(s):  return s.groupby(s.index.year).sum(min_count=300)
def monsoon(s):
    m = s[s.index.month.isin([6, 7, 8, 9])]
    return m.groupby(m.index.year).sum(min_count=100)
display(Markdown("✅ **Ready.** Tools loaded. Now run **Step 1** to choose your data."))

In [ ]:
def illustrative_three_sources(year=2021):
    """Illustrative only: a gauge record and how a grid cell and a satellite might see the same rain."""
    g = sample[str(year)].copy()
    rng = np.random.default_rng(7)
    grid = (0.55 * g + 0.45 * g.rolling(3, center=True, min_periods=1).mean() * 0.97).round(1)
    sat = g.rolling(2, min_periods=1).mean() * 1.03
    sat[(g > 0) & (g < 3) & (rng.random(len(g)) < 0.5)] = 0.0       # misses some light rain
    sat = (sat + np.where(rng.random(len(g)) < 0.03, rng.gamma(1, 3, len(g)), 0)).round(1)  # some false rain
    return pd.DataFrame({"Rain gauge (point)": g, "Gridded (area average)": grid, "Satellite estimate": sat})
three = illustrative_three_sources(2021)

---
## Part 1 · A point, an area, a view from space
- A **rain gauge** catches rain in a funnel about 20 cm wide. It is the most direct measurement, but only for that spot.
- A **gridded** dataset combines many gauges into an **average** for each square of a grid (for IMD, about 27 km × 27 km).
  Averaging smooths out the heaviest local bursts.
- A **satellite** estimate infers rain from clouds and microwaves seen from space. It covers everywhere, including oceans
  and forests, but it can miss light rain and sometimes reports rain that never reached the ground.
- A **reanalysis** (like ERA5) uses a weather model guided by observations to fill the whole globe consistently.

---
## Part 2 · Same year, three records (illustrative)
Below, one year of a gauge record, and how a grid cell and a satellite *might* see the same rain.
*These three series are illustrative, built from the sample district to show typical differences.*

In [ ]:
#@title ▶ Compare the three records { run: "auto" }
view = "Running total through the year" #@param ["Running total through the year", "Daily values in the wettest month", "Summary table"]
if view == "Running total through the year":
    ax = three.cumsum().plot(color=[BLUE, ORANGE, TEAL], lw=2)
    ax.set_ylabel("Running total (mm)"); ax.set_title("Same rain, three records: running total through 2021")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b")); plt.tight_layout(); plt.show()
elif view == "Daily values in the wettest month":
    m = three["Rain gauge (point)"].resample("MS").sum().idxmax()
    sub = three[m:m + pd.offsets.MonthEnd(0)]
    ax = sub.plot(kind="bar", color=[BLUE, ORANGE, TEAL], width=0.85, figsize=(12, 4.5))
    ax.set_xticklabels([d.strftime("%d") for d in sub.index], rotation=0); ax.set_ylabel("mm/day")
    ax.set_title(f"Daily rainfall in {m:%B %Y}: the grid smooths peaks, the satellite smears and misses")
    plt.tight_layout(); plt.show()
else:
    t = pd.DataFrame({"Year total (mm)": three.sum().round(0), "Wettest day (mm)": three.max().round(1),
                      "Rainy days (≥2.5 mm)": (three >= 2.5).sum()})
    display(t)

### What to notice
- **Totals are similar; peaks are not.** Averaging and smoothing lower the wettest day.
- The satellite **misses some light rain** and **adds some false rain**, so its rainy-day count differs.
- For floods (peaks), a nearby gauge is best. For water balance (totals over a basin), grids are excellent.

---
## Part 3 · Two real global datasets for your place
If you chose **My place** in Step 1, press ▶ to download the same year from two real datasets:
**NASA POWER** and **ERA5**. If you are using the sample district, this cell explains how to try it.

In [ ]:
#@title ▶ Step 1 · Choose your data { run: "auto" }
data_source = "Sample district (works offline)" #@param ["Sample district (works offline)", "My place (live download from NASA POWER)"]
place = "Mysuru, Karnataka" #@param {type:"string"}
latitude = 0 #@param {type:"number"}
longitude = 0 #@param {type:"number"}
rain, label, LAT, LON = load_rain(data_source, place, latitude, longitude)
display(Markdown(f"**Data:** {label} · {rain.index.min():%d %b %Y} to {rain.index.max():%d %b %Y} · "
                 f"{len(rain):,} days"))

In [ ]:
#@title ▶ NASA POWER vs ERA5 for one year { run: "auto" }
year = 2021 #@param {type:"slider", min:1981, max:2024, step:1}
if LAT is None:
    display(Markdown("ℹ️ You are using the sample district. In **Step 1**, choose *My place*, type a town or district, and run this cell again."))
else:
    try:
        p = rain[str(year)]
        e = fetch_era5(LAT, LON, f"{year}-01-01", f"{year}-12-31")
        both = pd.DataFrame({"NASA POWER": p, "ERA5": e}).dropna()
        ax = both.cumsum().plot(color=[ORANGE, GREY], lw=2)
        ax.set_ylabel("Running total (mm)"); ax.set_title(f"{label.split(' · ')[0]}: running total in {year}")
        plt.tight_layout(); plt.show()
        display(pd.DataFrame({"Year total (mm)": both.sum().round(0), "Wettest day (mm)": both.max().round(1),
                              "Rainy days (≥2.5 mm)": (both >= 2.5).sum()}))
    except Exception as ex:
        display(Markdown(f"⚠️ ERA5 download failed ({type(ex).__name__}). Try again in a minute."))

---
## Part 4 · Choosing a source

| If you need… | Prefer | Because |
|---|---|---|
| The official seasonal verdict for a district | IMD district rainfall | It is the reference others will quote |
| The peak of a local storm | The nearest gauge (state or IMD) | Grids and satellites smooth peaks |
| A long, gap-free series anywhere | IMD gridded, NASA POWER or ERA5 | No missing days, consistent method |
| Rain over sea, forest or a border area | Satellite (GPM) | Gauges are rare or absent |

Whatever you choose, **say which source you used**. Your capstone dashboard will carry a source note.

### ⏭️ Next
Open the **Practice activity** and compare the three records yourself.